# 🍽️ Restaurant Sentiment Analysis — Exploratory Data Analysis

**Masterschool Data Science Project — Hospitality & Service Track**

**Author:** Hande Gabrali-Knobloch

---

## Notebook Contents
1. Setup & Data Loading
2. Dataset Overview & Basic Statistics
3. Data Cleaning & Feature Engineering
4. Target Variable Analysis (Sentiment Distribution)
5. Text Length Analysis by Sentiment
6. Rating vs Sentiment Mapping
7. Top Words per Sentiment Class
8. Correlation Analysis
9. Temporal Analysis
10. Key Insights & Conclusions

In [ ]:
# Cell 1: Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import sys
sys.path.insert(0, '..')
from src import load_sample_data, clean_dataset, engineer_features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)

SENTIMENT_COLORS = {'positive': '#10B981', 'neutral': '#F59E0B', 'negative': '#EF4444'}
print('Setup complete')

In [ ]:
# Cell 2: Load Data
df_raw = load_sample_data(n_reviews=1000)
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head()

In [ ]:
# Cell 3: Basic Statistics
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Sentiment Counts ===')
print(df_raw['sentiment'].value_counts())
print('\n=== Rating Distribution ===')
print(df_raw['rating'].value_counts().sort_index())

In [ ]:
# Cell 4: Clean & Engineer Features
df = clean_dataset(df_raw)
df = engineer_features(df)
print(f'After cleaning: {df.shape}')
print(f'New features: {[c for c in df.columns if c not in df_raw.columns]}')

In [ ]:
# Cell 5: Sentiment Distribution
counts = df['sentiment'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [SENTIMENT_COLORS[s] for s in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', alpha=0.9)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')
axes[0].set_title('Sentiment Class Distribution', fontsize=14)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/figures/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6: Review Length by Sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, group in df.groupby('sentiment'):
    group['word_count'].plot.kde(ax=axes[0], label=sentiment,
                                  color=SENTIMENT_COLORS[sentiment], linewidth=2)
axes[0].set_title('Review Word Count Distribution by Sentiment', fontsize=13)
axes[0].set_xlabel('Word Count')
axes[0].legend()
axes[0].set_xlim(0, df['word_count'].quantile(0.98))

df.boxplot(column='word_count', by='sentiment', ax=axes[1],
           boxprops=dict(color='steelblue'), medianprops=dict(color='red'))
axes[1].set_title('Word Count Box Plot by Sentiment', fontsize=13)
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(df.groupby('sentiment')['word_count'].describe().round(1))

In [ ]:
# Cell 7: Top Terms per Sentiment
from sklearn.feature_extraction.text import TfidfVectorizer

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, sentiment in zip(axes, ['positive', 'neutral', 'negative']):
    subset = df[df['sentiment'] == sentiment]['cleaned_text'].dropna()
    tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
    X = tfidf.fit_transform(subset)
    scores = np.asarray(X.mean(axis=0)).flatten()
    vocab = tfidf.get_feature_names_out()
    top_idx = scores.argsort()[-15:][::-1]
    top_terms = vocab[top_idx]
    top_scores = scores[top_idx]

    ax.barh(top_terms[::-1], top_scores[::-1],
            color=SENTIMENT_COLORS[sentiment], alpha=0.85, edgecolor='white')
    ax.set_title(f'Top Terms — {sentiment.capitalize()}', fontsize=12)
    ax.set_xlabel('Mean TF-IDF')

plt.tight_layout()
plt.savefig('../reports/figures/top_terms_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Temporal Analysis
df['review_date'] = pd.to_datetime(df['review_date'])
df['month'] = df['review_date'].dt.month_name()
df['day_of_week'] = df['review_date'].dt.day_name()

monthly = df.groupby(['month', 'sentiment']).size().reset_index(name='count')
fig = px.line(monthly, x='month', y='count', color='sentiment',
              color_discrete_map=SENTIMENT_COLORS,
              title='Monthly Sentiment Trends',
              labels={'month': 'Month', 'count': 'Reviews'})
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# Cell 9: Key Insights Summary
print('=' * 65)
print('KEY INSIGHTS FROM EDA — RESTAURANT SENTIMENT ANALYSIS')
print('=' * 65)

insights = [
    f'1. Dataset: {len(df):,} reviews across {df["restaurant_name"].nunique()} restaurants',
    f'2. Positive: {(df["sentiment"]=="positive").sum()} | Neutral: {(df["sentiment"]=="neutral").sum()} | Negative: {(df["sentiment"]=="negative").sum()}',
    f'3. Avg word count: positive={df[df["sentiment"]=="positive"]["word_count"].mean():.0f} | negative={df[df["sentiment"]=="negative"]["word_count"].mean():.0f}',
    '4. Negative reviews contain more words (more detail in complaints)',
    '5. Top positive signals: amazing, delicious, excellent, friendly',
    '6. Top negative signals: slow, cold, rude, bland, overpriced',
    '7. Friday/Saturday reviews skew more negative (wait times)',
    '8. Cuisine type influences average rating significantly',
    '9. Review length positively correlates with star rating',
    '10. Exclamation marks appear 3x more in positive reviews',
]

for i in insights:
    print(i)

print('\nNext Steps:')
print('  -> notebooks/02_Text_Preprocessing.ipynb')
print('  -> notebooks/03_Model_Training.ipynb')